# Model Family Comparison (Analysis E)

## Protocol

**Status:** written on 2026-10-22, before any analysis code in this notebook was run. Under the project's commit policy it is not committed separately before execution; it is committed with the completed work package. Before execution the code was checked in two ways, neither of which reveals a result for any candidate:

- a smoke test with **randomly permuted grades** on two outer folds (macro F1 about 0.36, chance level);
- a reproduction pre-check that refit only the ECD tree and notebook 13's Random Forests on two outer folds and matched their published scores exactly.

### 1. Framing

From notebook 06 onward, only Random Forests were evaluated. The model family comparison in notebook 05 used folds that ignored donors (`CLAIMS_AUDIT.md`, T1). This analysis asks:

> **Can another model family, tuned the same way, grade better than the tuned Random Forest hybrid served by the application?**

It is a **follow-up**, not an independent confirmation:

- The outer folds are the same donor-grouped folds used in notebooks 09–13. They have influenced earlier decisions (cohort, comparators, PCA size, the application model).
- The question was raised after notebook 13's results were known.
- Independent confirmation requires new tissues.

Existing results are not changed or re-run, except for the reproduction checks in Section 4.

### 2. Cohort

As notebook 13, unchanged.

| Set | Use | Expected |
|---|---|---|
| Primary cohort K3: non-2023, image-matched, year from `Tissue_ID` | Training and inner tuning | 551 tissues, 312 donors |
| ECD evaluation set K4: K3 with ECD | Outer-fold evaluation of all models; training of the ECD tree | 470 tissues, 280 donors; A+ 244, A 159, B 67 |

The notebook asserts these counts and stops if they differ.

### 3. Validation design

As notebook 13, unchanged:

- **Outer folds:** `Fold_r0` … `Fold_r4` from `donor_grouped_folds.csv`, restricted to K3 (25 outer folds).
- **Inner folds:** within each outer training fold, `StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)`, grouped by inferred donor and stratified by grade. Every inner validation fold must contain all three grades.
- **Inner selection metric:** macro F1 averaged over the 3 inner validation folds. Ties go to the first setting in the grid order listed in Section 4.
- **Leakage controls:** all preprocessing is fitted on inner-training data for selection and on the full outer-training fold for the refit. The outer test fold is used only for the final prediction.

Outer folds are processed in parallel (one process per fold, one thread per process). This changes execution order only, not any fit.

### 4. Models

#### 4.1 Reference models (reproduced, not re-tuned)

| Model | How it is obtained | Reproduction check |
|---|---|---|
| ECD tree | Depth-2 tree on ECD, as notebooks 11 and 13 | Fold scores equal notebook 11 (`fitted_ecd_fold_scores.csv`) |
| Tuned RF tabular | Refit, in each outer fold, of the setting notebook 13 selected for that fold (`nested_tuning_selections.csv`) | Fold scores equal notebook 13 (`nested_tuning_fold_scores.csv`) |
| **Tuned RF hybrid** (deployed family) | As above | As above |

The reproduced models use exactly notebook 13's preprocessing. Each check requires a maximum absolute difference below 1e-9 in every metric. If a check fails, the notebook stops and the failure is recorded in the deviations log before any result is reported.

Reproducing the Random Forests also provides their out-of-fold predictions, which notebook 13 did not save, so that kappa and per-grade precision can be computed for every model.

#### 4.2 Candidate families (tuned in this notebook)

Each family is tuned in two arms: **tabular** (24 features) and **hybrid** (24 features + image), giving 8 candidates.

**Preprocessing:**

- **Tabular branch:** `ms.make_preprocessor()`, as in notebook 13: median imputation and standardisation of the measurements; one-hot encoding of slit-lamp findings, with "not recorded" as its own category.
- **Image branch (hybrid arms):** standardise the 2048 ResNet-50 embedding dimensions, PCA (random_state 42), then **standardise the PCA components**.

The final standardisation is new compared with notebook 13. Logistic regression and the RBF SVM depend on feature scale, and unscaled PCA components would dominate them. Tree models are unaffected by it. It is applied to all four families so that they share one pipeline.

PCA components for the hybrid arms are tuned over {8, 16, 32}, as in notebook 13.

**Search spaces** (grid order = tie-break order):

| Family | Fixed settings | Searched | Settings per PCA option |
|---|---|---|---:|
| Histogram gradient boosting (`HistGradientBoostingClassifier`) | `early_stopping=False`, `random_state=42` | (`learning_rate`, `max_iter`) ∈ {(0.03, 300), (0.1, 100)} × `max_leaf_nodes` {7, 15} × `min_samples_leaf` {20, 10} × `l2_regularization` {1.0, 0.0} × `class_weight` {balanced, None} | 32 |
| Logistic regression (`LogisticRegression`, multinomial, L2) | `solver="lbfgs"`, `max_iter=5000` | `C` {0.001, 0.01, 0.1, 1, 10} × `class_weight` {balanced, None} | 10 |
| RBF SVM (`SVC`) | `kernel="rbf"` | `C` {0.1, 1, 10, 100} × `gamma` {"scale", 0.01, 0.001} × `class_weight` {balanced, None} | 24 |
| Extra Trees (`ExtraTreesClassifier`) | `n_estimators=300`, `random_state=42` | Notebook 13's Random Forest grid: `min_samples_leaf` {5, 2, 1} × `max_depth` {15, None} × `max_features` {sqrt, 0.3} × `class_weight` {balanced, None} | 24 |

XGBoost and LightGBM are not installed. Histogram gradient boosting is scikit-learn's implementation of the same approach (histogram-based gradient-boosted trees).

Convergence warnings from logistic regression are counted and reported, not hidden.

### 5. Outcomes and comparisons

**Metrics** for every model on the K4 tissues of each outer test fold:

- macro F1 (**primary**);
- balanced accuracy;
- accuracy;
- grade B recall;
- quadratic-weighted Cohen's kappa, with the ordinal order B < A < A+.

Fold-level metrics are reported as mean ± SD over the 25 outer folds. Per-grade precision and recall, and kappa, are also reported pooled over repeats.

**Primary family P** (8 comparisons, Holm-corrected together): each candidate − tuned RF hybrid, macro F1.

**Secondary family S** (8 comparisons, Holm-corrected together): each candidate − ECD tree, macro F1.

**Secondary metrics:** for every comparison in P and S, differences in balanced accuracy, accuracy, grade B recall and kappa are reported unadjusted.

**Test:** the corrected repeated-CV t-test, as in notebooks 09–13 (Nadeau & Bengio). It uses the paired differences over the 25 outer folds, 24 degrees of freedom, and the K3 training and test fold sizes. The 95% CIs are not adjusted for multiplicity.

**Verdict for each primary comparison:**

| Verdict | Condition |
|---|---|
| Improved | difference > 0 and Holm p < 0.05 |
| Worse | difference < 0 and Holm p < 0.05 |
| Inconclusive | otherwise |

**Descriptive only (no tests):**

- each candidate's tabular arm against the tuned RF tabular model;
- the settings selected across outer folds.

### 6. Application model rule

Fixed here and applied mechanically.

A candidate **convincingly outperforms** the deployed model if, in family P:

- its macro F1 difference is > 0;
- its Holm p is < 0.05;
- its grade B recall difference is ≥ −0.02.

- If no candidate meets this, **the deployed tuned RF hybrid is retained.**
- If one or more meet it, the one with the highest mean outer-fold macro F1 is **recommended** as a replacement.

This notebook does not retrain, export or deploy any model. A recommended replacement would be a separate, documented task.

A higher point estimate without a convincing improvement does not change the deployed model.

### 7. Interpretation rules

- **A higher point estimate is not an improvement.** Only verdicts under Section 5 are reported as improvements or deteriorations.
- **An inconclusive result means the data cannot distinguish the models.** It does not show they are equivalent: no equivalence margin was set.
- **This is a follow-up on reused folds.** Any improvement found here would be a hypothesis for confirmation on new tissues, not an established result.
- **No re-runs to change results.** The notebook is executed once. Re-runs are allowed only to fix errors that do not change any choice in this protocol, and each is recorded in the deviations log.

# Analysis

Implements the protocol above without changes. Section numbers refer to the protocol.

In [1]:
import hashlib
import json
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits

warnings.filterwarnings("ignore", category=UserWarning)

SCRIPTS_DIR = str(Path("../../scripts").resolve())
sys.path.insert(0, SCRIPTS_DIR)
os.environ["PYTHONPATH"] = SCRIPTS_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")  # for worker processes
import modeling_setup as ms

SEED = ms.RANDOM_STATE
CLASSES = list(ms.CLASSES)
ORDINAL = ["B", "A", "A+"]  # kappa weights follow this order
ECD = "ECD_CD_cells_per_mm2"
TABULAR_FEATURES = list(ms.ALL_FEATURES)
IMAGE_DATA_DIR = ms.PROCESSED_DIR.parent.parent / "image" / "data" / "processed"
EMBEDDING_PATH = IMAGE_DATA_DIR / "embeddings" / "resnet50_imagenet_fullres.npz"
NB11_SCORES_PATH = ms.PROCESSED_DIR / "fitted_ecd_fold_scores.csv"
NB13_SCORES_PATH = ms.PROCESSED_DIR / "nested_tuning_fold_scores.csv"
NB13_SELECTIONS_PATH = ms.PROCESSED_DIR / "nested_tuning_selections.csv"
REVIEW_IDS = ["NEB-2024-08-16259 R", "NEB-2025-09-17471 R"]
INNER_SPLITS = 3
N_WORKERS = min(12, os.cpu_count())
PCA_GRID = [8, 16, 32]
TREE_PARAMS = dict(max_depth=2, min_samples_leaf=10, class_weight="balanced", criterion="gini", random_state=SEED)

# Protocol Section 4.2. Grid order is the tie-break order.
GRIDS = {
    "HGB": [
        {"learning_rate": rate, "max_iter": iterations, "max_leaf_nodes": leaves,
         "min_samples_leaf": leaf, "l2_regularization": l2, "class_weight": weight}
        for rate, iterations in [(0.03, 300), (0.1, 100)]
        for leaves in [7, 15]
        for leaf in [20, 10]
        for l2 in [1.0, 0.0]
        for weight in ["balanced", None]
    ],
    "LR": [{"C": c, "class_weight": weight} for c in [0.001, 0.01, 0.1, 1, 10] for weight in ["balanced", None]],
    "SVM": [
        {"C": c, "gamma": gamma, "class_weight": weight}
        for c in [0.1, 1, 10, 100]
        for gamma in ["scale", 0.01, 0.001]
        for weight in ["balanced", None]
    ],
    "ET": [
        {"min_samples_leaf": leaf, "max_depth": depth, "max_features": features, "class_weight": weight}
        for leaf in [5, 2, 1]
        for depth in [15, None]
        for features in ["sqrt", 0.3]
        for weight in ["balanced", None]
    ],
}
assert {family: len(grid) for family, grid in GRIDS.items()} == {"HGB": 32, "LR": 10, "SVM": 24, "ET": 24}

FAMILY_NAMES = {"HGB": "Gradient boosting", "LR": "Logistic regression", "SVM": "RBF SVM", "ET": "Extra Trees"}
ARMS = {"tabular": [None], "hybrid": PCA_GRID}
CANDIDATES = [f"{FAMILY_NAMES[family]} {arm}" for family in GRIDS for arm in ARMS]
REFERENCES = ["ECD tree", "Tuned RF tabular", "Tuned RF hybrid"]
MODELS = REFERENCES + CANDIDATES

for path in [ms.DATA_PATH, ms.DONOR_FOLDS_PATH, EMBEDDING_PATH, NB11_SCORES_PATH, NB13_SCORES_PATH, NB13_SELECTIONS_PATH]:
    print(f"{path.name}: sha256 {hashlib.sha256(path.read_bytes()).hexdigest()[:16]}…")

cleaned_grading_dataset.xlsx: sha256 0ce9b384a51a4c47…
donor_grouped_folds.csv: sha256 755280ff9d7bd388…
resnet50_imagenet_fullres.npz: sha256 883d4a1723a17be7…
fitted_ecd_fold_scores.csv: sha256 fe0c10e5d3431501…
nested_tuning_fold_scores.csv: sha256 ad9536db279af708…
nested_tuning_selections.csv: sha256 7ff3077bc130fa81…


## Cohort (Protocol Section 2)

In [2]:
def normalize_id(value):
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

labelled = ms.load_labelled_data()
folds = ms.get_donor_folds(labelled)

manifest = pd.read_csv(IMAGE_DATA_DIR / "clean_image_manifest.csv")
manifest["Match_ID"] = manifest["Tissue_ID"].map(normalize_id)
manifest = manifest[~manifest["Match_ID"].isin({normalize_id(i) for i in REVIEW_IDS})]

labelled["Match_ID"] = labelled["Tissue_ID"].map(normalize_id)
labelled["Year"] = labelled["Tissue_ID"].str.extract(r"NEB-\s*(\d{4})")[0].astype("Int64")

with_image = labelled.merge(manifest[["Match_ID"]], on="Match_ID", how="inner", validate="one_to_one")
cohort = (
    with_image[with_image["Year"].notna() & (with_image["Year"] != 2023)]
    .merge(folds, on="Tissue_ID", how="left", validate="one_to_one")
    .reset_index(drop=True)
)
has_ecd = cohort[ECD].notna().to_numpy()
evaluation = cohort[has_ecd]

primary_desc = {"tissues": len(cohort), "donors": cohort["Donor_Group"].nunique()}
eval_desc = {
    "tissues": len(evaluation), "donors": evaluation["Donor_Group"].nunique(),
    "grades": {k: int(v) for k, v in evaluation[ms.TARGET].value_counts().items()},
}
print("K3:", primary_desc, "| K4:", eval_desc)
assert primary_desc == {"tissues": 551, "donors": 312}, "K3 differs from protocol"
assert eval_desc == {"tissues": 470, "donors": 280, "grades": {"A+": 244, "A": 159, "B": 67}}, "K4 differs from protocol"

cached = np.load(EMBEDDING_PATH, allow_pickle=True)
row_of = {tissue_id: i for i, tissue_id in enumerate(cached["tissue_ids"])}
embeddings = cached["embeddings"][[row_of[t] for t in cohort["Tissue_ID"]]]
EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(embeddings.shape[1])]
X = pd.concat([
    cohort[TABULAR_FEATURES].reset_index(drop=True),
    pd.DataFrame(embeddings, columns=EMBEDDING_COLUMNS),
], axis=1)
y = cohort[ms.TARGET].to_numpy()
donors = cohort["Donor_Group"].to_numpy()
print("Feature matrix:", X.shape)

K3: {'tissues': 551, 'donors': 312} | K4: {'tissues': 470, 'donors': 280, 'grades': {'A+': 244, 'A': 159, 'B': 67}}
Feature matrix: (551, 2072)


## Models and tuning procedure (Protocol Sections 3–4)

In [3]:
TABULAR_PREPROCESSOR = ms.make_preprocessor()


def rf_preprocessor(n_components):
    # Notebook 13's preprocessing, unchanged (reference Random Forests only).
    branches = [("tabular", clone(TABULAR_PREPROCESSOR), TABULAR_FEATURES)]
    if n_components:
        branches.append(("image", Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=n_components, random_state=SEED)),
        ]), EMBEDDING_COLUMNS))
    return ColumnTransformer(branches, remainder="drop")


def candidate_preprocessor(n_components):
    # Protocol Section 4.2: as notebook 13, with the PCA components standardised.
    branches = [("tabular", clone(TABULAR_PREPROCESSOR), TABULAR_FEATURES)]
    if n_components:
        branches.append(("image", Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=n_components, random_state=SEED)),
            ("pca_scaler", StandardScaler()),
        ]), EMBEDDING_COLUMNS))
    return ColumnTransformer(branches, remainder="drop")


def make_model(family, setting):
    if family == "HGB":
        return HistGradientBoostingClassifier(early_stopping=False, random_state=SEED, **setting)
    if family == "LR":
        return LogisticRegression(solver="lbfgs", max_iter=5000, **setting)
    if family == "SVM":
        return SVC(kernel="rbf", **setting)
    if family == "ET":
        return ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=1, **setting)
    raise ValueError(family)


def make_rf(setting):
    # n_jobs does not affect a Random Forest's fit; one thread per outer-fold process.
    return RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=1, **setting)


def macro_f1(true, pred):
    return f1_score(true, pred, labels=CLASSES, average="macro", zero_division=0)


def tune(rows, family, pca_options, X, y, donors):
    # Select a setting by 3-fold donor-grouped inner CV on `rows` (positions in the cohort).
    candidates = [(k, s) for k in pca_options for s in GRIDS[family]]
    scores = np.zeros((INNER_SPLITS, len(candidates)))
    splitter = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
    for i, (inner_train, inner_val) in enumerate(splitter.split(rows, y[rows], groups=donors[rows])):
        tr, va = rows[inner_train], rows[inner_val]
        assert set(y[va]) == set(CLASSES), "An inner validation fold lacks a grade"
        assert set(donors[tr]).isdisjoint(donors[va]), "Inner donor leak"
        for k in pca_options:
            pre = candidate_preprocessor(k).fit(X.iloc[tr], y[tr])
            Z_tr, Z_va = pre.transform(X.iloc[tr]), pre.transform(X.iloc[va])
            for j, (kk, setting) in enumerate(candidates):
                if kk != k:
                    continue
                scores[i, j] = macro_f1(y[va], make_model(family, setting).fit(Z_tr, y[tr]).predict(Z_va))
    mean_scores = scores.mean(axis=0)
    best = int(np.argmax(mean_scores))  # first maximum = tie-break order
    return candidates[best], mean_scores[best]


def run_references(repeat, fold, train, test):
    # Reference models, fitted in the main process with default BLAS threading as in
    # notebooks 11 and 13: PCA rounding depends on the thread count (deviations log).
    eval_test = test[has_ecd[test]]
    predictions = {}
    tree = Pipeline([
        ("features", ColumnTransformer([("ecd", "passthrough", [ECD])], remainder="drop")),
        ("classifier", DecisionTreeClassifier(**TREE_PARAMS)),
    ]).fit(X.iloc[train[has_ecd[train]]], y[train[has_ecd[train]]])
    predictions["ECD tree"] = tree.predict(X.iloc[eval_test])
    for name in ["Tuned RF tabular", "Tuned RF hybrid"]:
        k, setting = rf_settings[(name, repeat, fold)]
        model = Pipeline([("features", rf_preprocessor(k)), ("classifier", make_rf(setting))])
        predictions[name] = model.fit(X.iloc[train], y[train]).predict(X.iloc[eval_test])
    return predictions


def run_outer_fold(repeat, fold, train, test, X, y, donors, has_ecd):
    # Candidate tuning for one outer fold, in a worker process limited to one thread.
    with threadpool_limits(1), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        assert set(donors[train]).isdisjoint(donors[test]), "Outer donor leak"
        eval_test = test[has_ecd[test]]
        assert set(y[eval_test]) == set(CLASSES), "An outer test fold lacks a grade"
        predictions, selections = {}, []

        for family in GRIDS:
            for arm, pca_options in ARMS.items():
                name = f"{FAMILY_NAMES[family]} {arm}"
                (k, setting), inner_score = tune(train, family, pca_options, X, y, donors)
                model = Pipeline([("features", candidate_preprocessor(k)), ("classifier", make_model(family, setting))])
                predictions[name] = model.fit(X.iloc[train], y[train]).predict(X.iloc[eval_test])
                selections.append({"Model": name, "Repeat": repeat, "Fold": fold, "PCA": k,
                                   "Setting": json.dumps(setting), "Inner macro F1": inner_score})

    convergence = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
    return {"repeat": repeat, "fold": fold, "eval_test": eval_test, "predictions": predictions,
            "selections": selections, "sizes": (len(train), len(test)), "convergence_warnings": convergence}


# Notebook 13's selected Random Forest setting for each outer fold (Protocol Section 4.1).
nb13_selections = pd.read_csv(NB13_SELECTIONS_PATH, dtype={"max_features": str})


def parse_rf_setting(row):
    features = row["max_features"]
    return (
        None if pd.isna(row["PCA"]) else int(row["PCA"]),
        {
            "min_samples_leaf": int(row["min_samples_leaf"]),
            "max_depth": None if pd.isna(row["max_depth"]) else int(row["max_depth"]),
            "max_features": features if features == "sqrt" else float(features),
            "class_weight": None if pd.isna(row["class_weight"]) else row["class_weight"],
        },
    )


RF_NAMES = {"Tuned tabular": "Tuned RF tabular", "Tuned hybrid": "Tuned RF hybrid"}
rf_settings = {
    (RF_NAMES[row["Arm"]], int(row["Repeat"]), int(row["Fold"])): parse_rf_setting(row)
    for row in nb13_selections.to_dict("records")
}
assert len(rf_settings) == 50, "Expected 25 outer folds × 2 Random Forest arms"

## Outer cross-validation (25 folds, run in parallel)

In [4]:
METRICS = {
    "Macro F1": macro_f1,
    "Balanced accuracy": balanced_accuracy_score,
    "Accuracy": accuracy_score,
    "Recall B": lambda t, p: recall_score(t, p, labels=["B"], average="macro", zero_division=0),
    "Kappa (quadratic)": lambda t, p: cohen_kappa_score(t, p, labels=ORDINAL, weights="quadratic"),
}

jobs = [
    (repeat, fold, train, test)
    for repeat in range(ms.N_REPEATS)
    for fold, (train, test) in enumerate(ms.iter_donor_folds(folds, cohort["Tissue_ID"], repeat))
]
assert len(jobs) == 25

start = time.time()
outputs = Parallel(n_jobs=N_WORKERS, backend="loky")(
    delayed(run_outer_fold)(repeat, fold, train, test, X, y, donors, has_ecd)
    for repeat, fold, train, test in jobs
)
for out, (repeat, fold, train, test) in zip(outputs, jobs):
    assert (out["repeat"], out["fold"]) == (repeat, fold)
    out["predictions"] = {**run_references(repeat, fold, train, test), **out["predictions"]}
print(f"Total time {(time.time() - start) / 60:.1f} min on {N_WORKERS} processes")

fold_rows, selection_rows, fold_sizes = [], [], []
oof = {(m, r): np.full(len(cohort), None, dtype=object) for m in MODELS for r in range(ms.N_REPEATS)}
for out in outputs:
    fold_sizes.append(out["sizes"])
    selection_rows.extend(out["selections"])
    for model, pred in out["predictions"].items():
        oof[(model, out["repeat"])][out["eval_test"]] = pred
        fold_rows.append({"Model": model, "Repeat": out["repeat"], "Fold": out["fold"], "N": len(out["eval_test"]),
                          **{m: fn(y[out["eval_test"]], pred) for m, fn in METRICS.items()}})

fold_scores = pd.DataFrame(fold_rows)
selections = pd.DataFrame(selection_rows)
N_TRAIN, N_TEST = np.mean(fold_sizes, axis=0)
print(f"Logistic regression convergence warnings: {sum(o['convergence_warnings'] for o in outputs)}")

Total time 41.9 min on 12 processes


Logistic regression convergence warnings: 0


### Reproduction checks (Protocol Section 4.1)

In [5]:
CHECKED = ["Macro F1", "Balanced accuracy", "Accuracy", "Recall B"]
for ours, path, theirs in [
    ("ECD tree", NB11_SCORES_PATH, "ECD tree"),
    ("Tuned RF tabular", NB13_SCORES_PATH, "Tuned tabular"),
    ("Tuned RF hybrid", NB13_SCORES_PATH, "Tuned hybrid"),
]:
    saved = pd.read_csv(path)
    a = fold_scores[fold_scores["Model"] == ours].set_index(["Repeat", "Fold"])[CHECKED].sort_index()
    b = saved[saved["Model"] == theirs].set_index(["Repeat", "Fold"])[CHECKED].sort_index()
    assert len(a) == len(b) == 25, f"{ours}: expected 25 folds"
    max_diff = (a - b).abs().to_numpy().max()
    print(f"{ours}: max absolute difference from {path.name} = {max_diff:.2e}")
    assert max_diff < 1e-9, f"{ours} not reproduced: stop and record in the deviations log"

ECD tree: max absolute difference from fitted_ecd_fold_scores.csv = 0.00e+00
Tuned RF tabular: max absolute difference from nested_tuning_fold_scores.csv = 0.00e+00
Tuned RF hybrid: max absolute difference from nested_tuning_fold_scores.csv = 5.55e-17


## Results (Protocol Section 5)

In [6]:
summary = fold_scores.groupby("Model")[list(METRICS)].agg(["mean", "std"]).reindex(MODELS)
print("Outer-fold metrics on K4 tissues, mean and SD over 25 folds:")
print(summary.round(3).to_string())

Outer-fold metrics on K4 tissues, mean and SD over 25 folds:
                            Macro F1        Balanced accuracy        Accuracy        Recall B        Kappa (quadratic)       
                                mean    std              mean    std     mean    std     mean    std              mean    std
Model                                                                                                                        
ECD tree                       0.796  0.038             0.768  0.043    0.804  0.031    0.790  0.117             0.771  0.056
Tuned RF tabular               0.805  0.034             0.781  0.037    0.809  0.030    0.793  0.115             0.783  0.043
Tuned RF hybrid                0.793  0.047             0.775  0.050    0.797  0.040    0.789  0.130             0.764  0.066
Gradient boosting tabular      0.785  0.046             0.770  0.046    0.782  0.038    0.806  0.114             0.753  0.048
Gradient boosting hybrid       0.792  0.034             0

In [7]:
def corrected_ttest(differences):
    d = np.asarray(differences, dtype=float)
    n = len(d)
    se = np.sqrt((1 / n + N_TEST / N_TRAIN) * d.var(ddof=1))
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), df=n - 1)
    half = stats.t.ppf(0.975, df=n - 1) * se
    return d.mean(), d.mean() - half, d.mean() + half, t, p


def holm(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[idx])
        adjusted[idx] = min(running, 1.0)
    return adjusted


wide = fold_scores.pivot_table(index=["Repeat", "Fold"], columns="Model", values=list(METRICS))
FAMILIES = {"P": "Tuned RF hybrid", "S": "ECD tree"}
rows = []
for family, baseline in FAMILIES.items():
    for i, candidate in enumerate(CANDIDATES, start=1):
        for metric in METRICS:
            d = wide[(metric, candidate)] - wide[(metric, baseline)]
            mean, low, high, t, p = corrected_ttest(d)
            rows.append({
                "ID": f"{family}{i}", "Family": family, "Candidate": candidate, "Baseline": baseline,
                "Metric": metric, "Model": wide[(metric, candidate)].mean(), "Baseline mean": wide[(metric, baseline)].mean(),
                "Difference": mean, "CI low": low, "CI high": high, "t": t, "p": p,
                "Folds improved": f"{(d > 0).sum()}/{len(d)}",
            })
results = pd.DataFrame(rows)
results["p (Holm)"] = np.nan
results["Verdict"] = ""
for family in FAMILIES:
    mask = (results["Family"] == family) & (results["Metric"] == "Macro F1")
    adjusted = holm(results.loc[mask, "p"])
    difference = results.loc[mask, "Difference"].to_numpy()
    results.loc[mask, "p (Holm)"] = adjusted
    results.loc[mask, "Verdict"] = np.where(
        (adjusted < 0.05) & (difference > 0), "Improved",
        np.where((adjusted < 0.05) & (difference < 0), "Worse", "Inconclusive"),
    )

COLUMNS = ["ID", "Candidate", "Model", "Baseline mean", "Difference", "CI low", "CI high", "p", "p (Holm)", "Folds improved", "Verdict"]
for family, baseline in FAMILIES.items():
    role = "Primary" if family == "P" else "Secondary"
    print(f"\n{role} family {family}: candidate − {baseline}, macro F1 (Holm across 8; CIs not multiplicity-adjusted)")
    print(results[(results["Family"] == family) & (results["Metric"] == "Macro F1")][COLUMNS].round(4).to_string(index=False))


Primary family P: candidate − Tuned RF hybrid, macro F1 (Holm across 8; CIs not multiplicity-adjusted)
ID                   Candidate  Model  Baseline mean  Difference  CI low  CI high      p  p (Holm) Folds improved      Verdict
P1   Gradient boosting tabular 0.7848         0.7934     -0.0085 -0.0462   0.0291 0.6437    1.0000           9/25 Inconclusive
P2    Gradient boosting hybrid 0.7921         0.7934     -0.0013 -0.0407   0.0381 0.9477    1.0000          11/25 Inconclusive
P3 Logistic regression tabular 0.6981         0.7934     -0.0953 -0.1588  -0.0318 0.0049    0.0274           1/25        Worse
P4  Logistic regression hybrid 0.6783         0.7934     -0.1150 -0.1873  -0.0427 0.0031    0.0220           1/25        Worse
P5             RBF SVM tabular 0.7152         0.7934     -0.0782 -0.1336  -0.0228 0.0076    0.0274           1/25        Worse
P6              RBF SVM hybrid 0.6813         0.7934     -0.1121 -0.1641  -0.0601 0.0002    0.0013           0/25        Worse
P7     

In [8]:
print("Secondary metrics (differences, unadjusted):")
secondary = results[results["Metric"] != "Macro F1"]
for family, baseline in FAMILIES.items():
    table = secondary[secondary["Family"] == family].pivot_table(
        index="Candidate", columns="Metric", values="Difference", sort=False)
    print(f"\nCandidate − {baseline}:")
    print(table.reindex(CANDIDATES).round(4).to_string())

print("\nSecondary-metric differences whose unadjusted 95% CI excludes zero:")
flagged = secondary[(secondary["CI low"] > 0) | (secondary["CI high"] < 0)]
print(flagged[["ID", "Candidate", "Baseline", "Metric", "Difference", "CI low", "CI high", "p"]].round(4).to_string(index=False)
      if len(flagged) else "None")

Secondary metrics (differences, unadjusted):

Candidate − Tuned RF hybrid:
Metric                       Balanced accuracy  Accuracy  Recall B  Kappa (quadratic)
Candidate                                                                            
Gradient boosting tabular              -0.0050   -0.0155    0.0174            -0.0110
Gradient boosting hybrid               -0.0047   -0.0088    0.0064            -0.0002
Logistic regression tabular            -0.0890   -0.0881   -0.0994            -0.0899
Logistic regression hybrid             -0.0984   -0.1021   -0.0761            -0.0996
RBF SVM tabular                        -0.0739   -0.0664   -0.1158            -0.0657
RBF SVM hybrid                         -0.1023   -0.0978   -0.1298            -0.1009
Extra Trees tabular                    -0.0745   -0.0674   -0.0974            -0.0630
Extra Trees hybrid                     -0.0551   -0.0428   -0.0724            -0.0279

Candidate − ECD tree:
Metric                       Balanced accu

### Descriptive results

In [9]:
print("Tabular arms against the tuned RF tabular model (mean fold-level macro F1 difference; descriptive, no tests):")
for family in GRIDS:
    name = f"{FAMILY_NAMES[family]} tabular"
    d = wide[("Macro F1", name)] - wide[("Macro F1", "Tuned RF tabular")]
    print(f"  {name}: {d.mean():+.4f} (higher in {(d > 0).sum()}/{len(d)} folds)")

Tabular arms against the tuned RF tabular model (mean fold-level macro F1 difference; descriptive, no tests):
  Gradient boosting tabular: -0.0201 (higher in 3/25 folds)
  Logistic regression tabular: -0.1068 (higher in 0/25 folds)
  RBF SVM tabular: -0.0897 (higher in 0/25 folds)
  Extra Trees tabular: -0.0957 (higher in 0/25 folds)


In [10]:
eval_rows = np.flatnonzero(has_ecd)
y_eval = np.tile(y[eval_rows], ms.N_REPEATS)
pooled = {m: np.concatenate([oof[(m, r)][eval_rows] for r in range(ms.N_REPEATS)]).astype(str) for m in MODELS}
assert all(len(p) == len(y_eval) and "None" not in set(p) for p in pooled.values())

per_grade = pd.concat({
    m: pd.DataFrame({
        "Precision": precision_score(y_eval, pooled[m], labels=CLASSES, average=None, zero_division=0),
        "Recall": recall_score(y_eval, pooled[m], labels=CLASSES, average=None, zero_division=0),
    }, index=CLASSES).stack()
    for m in MODELS
}, axis=1).T
per_grade["Kappa (quadratic)"] = [cohen_kappa_score(y_eval, pooled[m], labels=ORDINAL, weights="quadratic") for m in MODELS]
print("Per-grade precision and recall, and quadratic-weighted kappa, on K4 pooled over repeats:")
print(per_grade.round(3).to_string())

Per-grade precision and recall, and quadratic-weighted kappa, on K4 pooled over repeats:
                                   A+                A                B        Kappa (quadratic)
                            Precision Recall Precision Recall Precision Recall                  
ECD tree                        0.748  0.988     0.884  0.528     1.000  0.791             0.772
Tuned RF tabular                0.779  0.943     0.805  0.609     0.978  0.794             0.785
Tuned RF hybrid                 0.788  0.906     0.761  0.635     0.936  0.791             0.769
Gradient boosting tabular       0.784  0.870     0.719  0.636     0.928  0.809             0.756
Gradient boosting hybrid        0.777  0.892     0.733  0.625     0.982  0.797             0.767
Logistic regression tabular     0.752  0.816     0.593  0.552     0.808  0.690             0.677
Logistic regression hybrid      0.748  0.811     0.578  0.504     0.719  0.716             0.668
RBF SVM tabular                 0.775 

In [11]:
print("Selected settings across 25 outer folds (most frequent five per model):")
selections["PCA"] = selections["PCA"].map(lambda k: "none" if k is None or pd.isna(k) else str(int(k)))
for name in CANDIDATES:
    rows = selections[selections["Model"] == name]
    counts = (rows["PCA"] + " | " + rows["Setting"]).value_counts()
    print(f"\n{name}: {len(counts)} distinct settings; mean inner macro F1 {rows['Inner macro F1'].mean():.3f}")
    print(counts.head(5).rename("Folds").to_string())

Selected settings across 25 outer folds (most frequent five per model):

Gradient boosting tabular: 13 distinct settings; mean inner macro F1 0.729
none | {"learning_rate": 0.1, "max_iter": 100, "max_leaf_nodes": 15, "min_samples_leaf": 10, "l2_regularization": 0.0, "class_weight": null}    5
none | {"learning_rate": 0.03, "max_iter": 300, "max_leaf_nodes": 7, "min_samples_leaf": 10, "l2_regularization": 0.0, "class_weight": null}    4
none | {"learning_rate": 0.1, "max_iter": 100, "max_leaf_nodes": 7, "min_samples_leaf": 10, "l2_regularization": 0.0, "class_weight": null}     3
none | {"learning_rate": 0.1, "max_iter": 100, "max_leaf_nodes": 7, "min_samples_leaf": 10, "l2_regularization": 1.0, "class_weight": null}     2
none | {"learning_rate": 0.1, "max_iter": 100, "max_leaf_nodes": 15, "min_samples_leaf": 10, "l2_regularization": 1.0, "class_weight": null}    2

Gradient boosting hybrid: 23 distinct settings; mean inner macro F1 0.763
16 | {"learning_rate": 0.03, "max_iter": 300, "

## Application model rule (Protocol Section 6)

The rule is applied mechanically.

In [12]:
primary = results[(results["Family"] == "P") & (results["Metric"] == "Macro F1")].set_index("Candidate")
b_recall = results[(results["Family"] == "P") & (results["Metric"] == "Recall B")].set_index("Candidate")["Difference"]

convincing = []
for name in CANDIDATES:
    row = primary.loc[name]
    ok = bool(row["Difference"] > 0 and row["p (Holm)"] < 0.05 and b_recall[name] >= -0.02)
    if ok:
        convincing.append(name)
    print(f"{name}: macro F1 {row['Difference']:+.4f} [{row['CI low']:+.4f}, {row['CI high']:+.4f}], "
          f"Holm p = {row['p (Holm)']:.4f}, B recall difference {b_recall[name]:+.4f} -> "
          f"{'convincingly outperforms' if ok else 'does not convincingly outperform'} the tuned RF hybrid")

if convincing:
    recommended = max(convincing, key=lambda n: primary.loc[n, "Model"])
    print(f"\nRecommended replacement (not deployed by this notebook): {recommended}")
else:
    print("\nNo candidate convincingly outperforms the deployed model: the tuned RF hybrid is retained.")

Gradient boosting tabular: macro F1 -0.0085 [-0.0462, +0.0291], Holm p = 1.0000, B recall difference +0.0174 -> does not convincingly outperform the tuned RF hybrid
Gradient boosting hybrid: macro F1 -0.0013 [-0.0407, +0.0381], Holm p = 1.0000, B recall difference +0.0064 -> does not convincingly outperform the tuned RF hybrid
Logistic regression tabular: macro F1 -0.0953 [-0.1588, -0.0318], Holm p = 0.0274, B recall difference -0.0994 -> does not convincingly outperform the tuned RF hybrid
Logistic regression hybrid: macro F1 -0.1150 [-0.1873, -0.0427], Holm p = 0.0220, B recall difference -0.0761 -> does not convincingly outperform the tuned RF hybrid
RBF SVM tabular: macro F1 -0.0782 [-0.1336, -0.0228], Holm p = 0.0274, B recall difference -0.1158 -> does not convincingly outperform the tuned RF hybrid
RBF SVM hybrid: macro F1 -0.1121 [-0.1641, -0.0601], Holm p = 0.0013, B recall difference -0.1298 -> does not convincingly outperform the tuned RF hybrid
Extra Trees tabular: macro F1

In [13]:
fold_scores.to_csv(ms.PROCESSED_DIR / "model_family_fold_scores.csv", index=False)
results.to_csv(ms.PROCESSED_DIR / "model_family_results.csv", index=False)
selections.to_csv(ms.PROCESSED_DIR / "model_family_selections.csv", index=False)
per_grade.to_csv(ms.PROCESSED_DIR / "model_family_per_grade.csv")
pd.DataFrame({
    "Tissue_ID": np.tile(cohort["Tissue_ID"].to_numpy()[eval_rows], ms.N_REPEATS),
    "Repeat": np.repeat(np.arange(ms.N_REPEATS), len(eval_rows)),
    "Actual": y_eval,
    **pooled,
}).to_csv(ms.PROCESSED_DIR / "model_family_oof_predictions.csv", index=False)
print("Saved fold scores, comparisons, selections, per-grade metrics and out-of-fold predictions to", ms.PROCESSED_DIR)

Saved fold scores, comparisons, selections, per-grade metrics and out-of-fold predictions to C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed


## Deviations Log

Record any rerun or departure from the protocol here, with date and reason, before reporting results.

| Date | Change | Reason |
|---|---|---|
| 2026-10-22 | The first execution stopped at the reproduction check. The tuned RF hybrid differed from notebook 13 in one outer fold (repeat 1, fold 1; largest metric difference 0.018). The ECD tree and the RF tabular model matched exactly. No candidate result had been printed or saved. The three reference models are now fitted in the main process with default (multi-threaded) BLAS, as in notebook 13. Candidates are still tuned in parallel worker processes limited to one thread. Executed once more. | Diagnosed by refitting only the RF hybrid, whose scores were already published. With multi-threaded BLAS all 25 folds match notebook 13 exactly. Limited to one thread, the PCA's floating-point rounding changes, which alters predictions in that fold. Protocol Section 3's statement that parallel execution "changes execution order only, not any fit" was therefore incorrect for models with PCA. It affects only how the references are reproduced; no protocol choice changes. |